In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 28 samples and 13949 columns.
First few rows of the data:


,overall survival (days),vital status,sample_title,patient id,anti-pd-1 response,study site,gender,age (yrs),disease status,previous mapki,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,607.0,1,Pt1,Pt1,Progressive Disease,UCLA,F,66,M1b,N,...,6.945069,6.289998,7.130597,7.392799,5.839262,5.989320,7.615436,7.070199,5.974466,7.075712
1,927.0,0,Pt2,Pt2,Partial Response,UCLA,M,55,M1c,N,...,8.772441,7.548880,8.507697,9.111232,8.325919,7.400845,9.174003,8.206970,7.369614,8.626720
2,948.0,0,Pt4,Pt4,Partial Response,UCLA,M,62,M1c,Y,...,9.668285,9.305375,9.893391,10.050158,8.971447,8.294603,9.938084,9.148508,8.111535,9.433522
3,439.0,0,Pt5,Pt5,Partial Response,UCLA,M,61,M1c,N,...,4.576620,3.882534,4.736389,5.066071,3.423486,3.613893,5.227428,4.560005,3.456208,4.758636
4,882.0,0,Pt6,Pt6,Partial Response,UCLA,M,51,M1c,Y,...,9.515566,9.599917,10.313973,10.136349,8.642516,8.610092,10.228906,9.957816,8.859417,9.782276


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for overall survival (days) or vital status: 1
Number of samples with complete data for both variables: 27
Number of censored samples (event_var == 0.0): 15
Number of events (event_var == 1.0): 12
Censored ratio (among complete): 15 / 27 = 0.556 (55.6%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE78220/description.json
